<a href="https://colab.research.google.com/github/Seripro/c-learning/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun May 24 01:30:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile main.cu
#include <stdio.h>
#include <stdlib.h>

// ==========================================
// ★1. GPU側で実行される「カーネル（Kernel）」関数
// ==========================================
// __global__ をつけることで「CPUから呼び出せて、GPU側で動く関数」になる。
__global__ void vectorAdd(const double *A, const double *B, double *C, long long N) {
    // 【超重要】自分が「軍隊全体の何番目の作業員（スレッド）なのか」を計算する
    // blockDim.x : 1つの小隊の人数（スレッド数）
    // blockIdx.x : 自分が何番目の小隊（ブロック）か
    // threadIdx.x: 小隊の中での自分の出席番号
    long long i = blockDim.x * blockIdx.x + threadIdx.x;

    // 配列の範囲を超えないようにチェックして、自分の担当要素だけを足し算する！
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main(void) {
    long long N = 1000000;
    size_t size = sizeof(double) * N;

    // HPCのお作法：CPUのポインタには「h_（Host）」、GPUのポインタには「d_（Device）」をつけて区別する。
    // これをサボると、住所がごっちゃになって100%バグるぞ！
    double *h_A = (double*)malloc(size);
    double *h_B = (double*)malloc(size);
    double *h_C = (double*)malloc(size);

    // CPU側でのデータ初期化
    for (long long i = 0; i < N; i++) {
        h_A[i] = 1.0;
        h_B[i] = 2.0;
    }

    // GPU側の住所（ポインタ）を用意
    double *d_A, *d_B, *d_C;

    // ==========================================
    // ★2. GPUのメモリ（工場）を直接確保する
    // ==========================================
    // C言語の malloc の代わりに、CUDA専用の cudaMalloc を使う。
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // ==========================================
    // ★3. データをCPUからGPUへ手動で「転送（行き）」する
    // ==========================================
    // cudaMemcpy(コピー先, コピー元, サイズ, 転送の向き)
    // cudaMemcpyHostToDevice は「CPU（Host）からGPU（Device）へ送れ」という命令。
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // ==========================================
    // ★4. GPUの作業員（スレッド・ブロック）の配置を設計する
    // ==========================================
    int threadsPerBlock = 256; // 1つの小隊（ブロック）に256人配置する
    // 全体で N 個の計算があるので、必要な小隊の数を計算する（切り上げ計算）
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // ==========================================
    // ★5. GPUカーネルの起動（小学生たち、一斉に計算開始！）
    // ==========================================
    // <<<小隊の数, 1小隊の人数>>> という特別な記法（トリプルアングルブラケット）を使う！
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // ==========================================
    // ★6. データをGPUからCPUへ手動で「転送（帰り）」する
    // ==========================================
    // cudaMemcpyDeviceToHost は「GPU（Device）からCPU（Host）へ戻せ」という命令。
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // 結果の確認
    printf("C[0] = %f\n", h_C[0]);

    // ==========================================
    // ★7. お片付け（GPUとCPUの両方のメモリを解放する）
    // ==========================================
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing main.cu


In [ ]:
!wget -q https://developer.download.nvidia.com/hpc-sdk/25.1/nvhpc_2025_251_Linux_x86_64_cuda_multi.tar.gz
!tar -xzf nvhpc_2025_251_Linux_x86_64_cuda_multi.tar.gz

In [ ]:
!nvcc main.cu -o main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
cc1plus: fatal error: main.cu: No such file or directory
compilation terminated.


In [ ]:
%%writefile main.cu
#include <stdio.h>
#include <stdlib.h>

// ==========================================
// ★1. GPU側で実行される「カーネル（Kernel）」関数
// ==========================================
// __global__ をつけることで「CPUから呼び出せて、GPU側で動く関数」になる。
__global__ void vectorAdd(const double *A, const double *B, double *C, long long N) {
    // 【超重要】自分が「軍隊全体の何番目の作業員（スレッド）なのか」を計算する
    // blockDim.x : 1つの小隊の人数（スレッド数）
    // blockIdx.x : 自分が何番目の小隊（ブロック）か
    // threadIdx.x: 小隊の中での自分の出席番号
    long long i = blockDim.x * blockIdx.x + threadIdx.x;

    // 配列の範囲を超えないようにチェックして、自分の担当要素だけを足し算する！
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main(void) {
    long long N = 1000000;
    size_t size = sizeof(double) * N;

    // HPCのお作法：CPUのポインタには「h_（Host）」、GPUのポインタには「d_（Device）」をつけて区別する。
    // これをサボると、住所がごっちゃになって100%バグるぞ！
    double *h_A = (double*)malloc(size);
    double *h_B = (double*)malloc(size);
    double *h_C = (double*)malloc(size);

    // CPU側でのデータ初期化
    for (long long i = 0; i < N; i++) {
        h_A[i] = 1.0;
        h_B[i] = 2.0;
    }

    // GPU側の住所（ポインタ）を用意
    double *d_A, *d_B, *d_C;

    // ==========================================
    // ★2. GPUのメモリ（工場）を直接確保する
    // ==========================================
    // C言語の malloc の代わりに、CUDA専用の cudaMalloc を使う。
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // ==========================================
    // ★3. データをCPUからGPUへ手動で「転送（行き）」する
    // ==========================================
    // cudaMemcpy(コピー先, コピー元, サイズ, 転送の向き)
    // cudaMemcpyHostToDevice は「CPU（Host）からGPU（Device）へ送れ」という命令。
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // ==========================================
    // ★4. GPUの作業員（スレッド・ブロック）の配置を設計する
    // ==========================================
    int threadsPerBlock = 256; // 1つの小隊（ブロック）に256人配置する
    // 全体で N 個の計算があるので、必要な小隊の数を計算する（切り上げ計算）
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // ==========================================
    // ★5. GPUカーネルの起動（小学生たち、一斉に計算開始！）
    // ==========================================
    // <<<小隊の数, 1小隊の人数>>> という特別な記法（トリプルアングルブラケット）を使う！
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // ==========================================
    // ★6. データをGPUからCPUへ手動で「転送（帰り）」する
    // ==========================================
    // cudaMemcpyDeviceToHost は「GPU（Device）からCPU（Host）へ戻せ」という命令。
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // 結果の確認
    printf("C[0] = %f\n", h_C[0]);

    // ==========================================
    // ★7. お片付け（GPUとCPUの両方のメモリを解放する）
    // ==========================================
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing main.cu


In [ ]:
!nvcc main.cu -o main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./main

C[0] = 3.000000
